In [ ]:
#==============================================================================
# Loading the data
#==============================================================================

import pandas as pd

url = "https://raw.githubusercontent.com/Amirhossein-momeni/AmirGeek_ML_Datasets/refs/heads/main/DataSet/01-Ad.csv"
df = pd.read_csv(url)

In [ ]:
df.head()

In [ ]:
#==============================================================================
# Defining features and target
#==============================================================================

X = df.drop(columns=["UserID", "Purchased"])
y = df["Purchased"]

In [ ]:
#==============================================================================
# Preprocessing 
#==============================================================================

# Train / Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["Age", "EstimatedSalary"]
cat_cols = ["Gender", "Type"]

prep_num_only = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
])

prep_with_cat = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first"), cat_cols),
])

In [ ]:
#==============================================================================
# Building the model + choosing the feature set with CV
#==============================================================================

from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

model_num = Pipeline([("prep", prep_num_only), ("knn", KNeighborsClassifier(n_neighbors=5))])
model_cat = Pipeline([("prep", prep_with_cat), ("knn", KNeighborsClassifier(n_neighbors=5))])

for name, m in [("numeric only", model_num), ("numeric + categorical", model_cat)]:
    score = cross_val_score(m, X_train, y_train, cv=5, scoring="f1").mean()
    print(name, round(score, 3))
# numeric only          -> 0.875
# numeric + categorical -> 0.837

In [ ]:
#==============================================================================
# Experiment: what happens WITHOUT scaling?
#==============================================================================

import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

prep_raw = ColumnTransformer([("num", "passthrough", num_cols)])
model_raw = Pipeline([("prep", prep_raw), ("knn", KNeighborsClassifier(n_neighbors=5))])

model_raw.fit(X_train, y_train)
y_pred_raw = model_raw.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_raw)}\n")      # 0.7625
print(f"Cross_validation F1: {cross_val_score(model_raw, X_train, y_train, cv=5, scoring='f1').mean()}\n")  # ~0.72

print(f"Confusion matrix:\n{confusion_matrix(y_test, y_pred_raw)}")
cm = confusion_matrix(y_test, y_pred_raw, normalize="true")
ConfusionMatrixDisplay(cm).plot(
    cmap="Blues", values_format=".2f"
)
plt.show()

In [ ]:
#==============================================================================
# How does k affect overfitting? (train accuracy vs CV score)
#==============================================================================

for k in [1, 3, 5, 7, 11, 15, 21, 31, 51]:
    m = Pipeline([("prep", prep_num_only), ("knn", KNeighborsClassifier(n_neighbors=k))])
    cv_f1 = cross_val_score(m, X_train, y_train, cv=5, scoring="f1").mean()
    train_acc = m.fit(X_train, y_train).score(X_train, y_train)
    print(k, round(train_acc, 3), round(cv_f1, 3))

In [ ]:
#==============================================================================
# Training + evaluation (baseline, k=5)
#==============================================================================

from sklearn.metrics import classification_report

model = model_num
model.fit(X_train, y_train)          # KNN "training" = just storing the data

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}\n")          # 0.8875
print(f"Classification Report:\n\n{classification_report(y_test, y_pred)}")

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}\n")
cm = confusion_matrix(y_test, y_pred, normalize="true")
ConfusionMatrixDisplay(cm).plot(
    cmap="Blues", values_format=".2f"
)
plt.show()

In [ ]:
#==============================================================================
# Looking inside: who are the neighbors of one customer?
#==============================================================================

one_customer = X_test.iloc[[0]]                       # Female, type A, age 68, salary 23.6M
Z = model.named_steps["prep"].transform(one_customer)  # scale the same way as training
distances, idx = model.named_steps["knn"].kneighbors(Z)

print(y_train.iloc[idx[0]].tolist())    # labels of the neighbors

In [ ]:
#==============================================================================
# Hyperparameter tuning
#==============================================================================

from sklearn.model_selection import GridSearchCV

param_grid = {
    "knn__n_neighbors": list(range(1, 32, 2)),   # odd k avoids ties in binary problems
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
}

search = GridSearchCV(model_num, param_grid, cv=5, scoring="f1", n_jobs=-1)
search.fit(X_train, y_train)

print(search.best_params_)   # {'knn__n_neighbors': 7, 'knn__p': 1, 'knn__weights': 'uniform'}
print(search.best_score_)    # ~0.879

In [ ]:
#==============================================================================
# Final evaluation (once!) + probabilities
#==============================================================================

best_model = search.best_estimator_
y_pred_final = best_model.predict(X_test)

print(accuracy_score(y_test, y_pred_final))      # 0.925
print(confusion_matrix(y_test, y_pred_final))    # [[48  3] [ 3 26]]
print(classification_report(y_test, y_pred_final))

proba = best_model.predict_proba(X_test)[:, 1]
print(proba[:3].round(3))                        # [0.857 0.    0.   ]
print(y_test.iloc[:3].tolist())                  # [1, 0, 0]